# FuzNex Kaggle video pipeline

Executable, restart-safe replay of the shared GitHub AI package: YOLO11n → SCRFD → optional local SFace. It does **not** copy inference code into the notebook. Enable a Kaggle GPU before running.

Use a consented private video or an openly licensed clip. Public clips test detection only. Recognition is not verified until a genuine consented test succeeds. Never publish a gallery, embeddings, enrollment photos, or identifiable outputs.


In [ ]:
# Change to a reviewed commit SHA for an experiment; the checkout is detached/read-only.
FUZNEX_REF = 'integrate/portable-ai'
!rm -rf /kaggle/working/fuznex-smart-surveillance
!git clone --filter=blob:none --no-checkout https://github.com/harrsh-here/fuznex-smart-surveillance.git /kaggle/working/fuznex-smart-surveillance
%cd /kaggle/working/fuznex-smart-surveillance
!git fetch origin $FUZNEX_REF --depth 1
!git checkout --detach FETCH_HEAD
!git rev-parse HEAD
!git status --short


In [ ]:
# Run after each notebook restart. Kaggle's CUDA-enabled PyTorch is used by Ultralytics.
!python -m pip install -q pytest -r ai/requirements/yolo.txt
!python -m unittest discover -s tests -v


## Attach private Kaggle inputs

Attach a **video** dataset and a reviewed **models** dataset. Optionally attach a private, consented gallery dataset only when you deliberately enable SFace. The repository never downloads models. Review `ai/model-manifest.json`: SCRFD is hash-checked; YOLO11n has no pinned hash in this manifest, so record its provenance and SHA-256 outside Git.


In [ ]:
from pathlib import Path
VIDEO = Path('/kaggle/input/YOUR_PRIVATE_VIDEO/clip.mp4')
MODELS = Path('/kaggle/input/YOUR_REVIEWED_MODELS')
YOLO = MODELS / 'yolo11n.pt'
SCRFD = MODELS / 'det_500m.onnx'
SFACE = MODELS / 'face_recognition_sface_2021dec.onnx'
# Keep this False unless the gallery and input video are consented and private.
PRIVATE_RECOGNITION = False
GALLERY = Path('/kaggle/input/YOUR_PRIVATE_GALLERY/private-gallery.json')
OUTPUT_ROOT = Path('/kaggle/working/fuznex-sessions')
assert VIDEO.is_file() and YOLO.is_file() and SCRFD.is_file(), 'Set the attached video and approved model paths.'
if PRIVATE_RECOGNITION:
    assert SFACE.is_file() and GALLERY.is_file(), 'Private recognition requires the matching SFace model and a consented private gallery.'


In [ ]:
# One reusable experiment configuration. Every run creates a unique spectra-* session directory.
import subprocess
command = ['python', '-m', 'ai.video_replay', '--input', str(VIDEO), '--output-dir', str(OUTPUT_ROOT), '--yolo', str(YOLO), '--scrfd', str(SCRFD), '--device', 'cuda', '--input-size', '320', '--confidence', '0.25']
if PRIVATE_RECOGNITION:
    command += ['--sface', str(SFACE), '--private-gallery', str(GALLERY), '--private-recognition']
completed = subprocess.run(command, text=True, capture_output=True)
print(completed.stdout)
if completed.returncode:
    raise RuntimeError(completed.stderr or 'Replay was blocked or failed; inspect the JSON above.')


In [ ]:
# H.264 annotated video preview and privacy-safe metrics/session logs.
from IPython.display import Video, display
sessions = sorted(OUTPUT_ROOT.glob('spectra-*'))
assert sessions, 'No session was produced.'
session = sessions[-1]
display(Video(str(session / 'annotated_h264.mp4'), embed=True, width=900))
print((session / 'session-summary.json').read_text())
print((session / 'provenance.json').read_text())
# detections.csv/events.jsonl exclude names, identities, gallery contents, and embeddings.


## Interpretation and restart policy

The overlay may show Known, Unknown, Uncertain, or Unevaluable face state. It never draws a person-to-face relationship, an identity name, or a similarity score. Record the detached commit SHA, source-video consent/license, model hashes, Kaggle accelerator, and parameters with each run. FPS and latency represent Kaggle decode/inference/render work—not camera or network latency, and not local-Windows performance.

After a restart, rerun the checkout and dependency cells. Existing `/kaggle/working/fuznex-sessions/spectra-*` directories are never overwritten; download needed artifacts before ending the Kaggle session.
